# GRACE H3-v2 — CPU/RAM chạy local, không cần Colab

Notebook này giữ tên cũ để không làm mất đường dẫn bạn đang dùng. Chạy offline trên máy local/WSL, không mount Drive, không upload/download. CPU stages nhận input từ **bundles/** và ghi output/checkpoint trở lại thư mục đó.

Đặt PHASE='base' cho lượt đầu. Sau khi full-pipeline chạy GRACE validation và tạo meta-inputs, đổi thành PHASE='meta'. Quay lại full-pipeline để tiếp tục các bước còn lại.

LLM chỉ được gọi trong notebook full-pipeline; gọi Vertex cần mạng hoặc exact response cache đã có.


In [ ]:
from pathlib import Path
import os,sys,json,zipfile,hashlib,shutil,subprocess,time
PHASE='base'  # Lượt sau khi validation complete: đổi thành 'meta'
INSTALL_DEPENDENCIES=False  # Giữ False để chạy offline với môi trường đã cài sẵn
if PHASE not in ('base','meta'): raise ValueError('PHASE phải là base hoặc meta')
candidates=[Path.cwd(),*Path.cwd().parents,Path('/mnt/d/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble'),Path('D:/JUNIOR/LabThayTho/GRACE-H3-BenchmarkEnsemble')]
if os.environ.get('GRACE_H3V2_ROOT'): candidates.insert(0,Path(os.environ['GRACE_H3V2_ROOT']))
ROOT=next((p.resolve() for p in candidates if (p/'configs/benchmark_v2.json').exists()),None)
if ROOT is None: raise RuntimeError('Không tìm thấy worktree GRACE-H3-BenchmarkEnsemble; mở notebook trong thư mục project hoặc đặt GRACE_H3V2_ROOT')
BUNDLES=ROOT/'bundles'; BUNDLES.mkdir(exist_ok=True)
INPUT=BUNDLES/f'grace-h3v2-{PHASE}-inputs.zip'
RUNTIME=BUNDLES/'grace-h3-joern-runtime.zip'
CHECKPOINT=BUNDLES/f'grace-h3v2-{PHASE}-checkpoint.zip'
if not INPUT.exists(): raise FileNotFoundError(f'{INPUT} chưa có. Mở notebooks/full-pipeline.ipynb local và chạy tới handoff {PHASE}.')
if not RUNTIME.exists(): raise FileNotFoundError(f'Thiếu Joern runtime local: {RUNTIME}')
print('PHASE:',PHASE,'ROOT:',ROOT,'Bundles local:',BUNDLES)


## Nhập input local

Notebook chỉ đọc bundle trong project. Không upload, mount Drive hoặc chép file sang dịch vụ ngoài. Importer kiểm tra checksum và phase trước khi ghi vào workspace.


In [ ]:
with zipfile.ZipFile(INPUT) as z:
    m=json.loads(z.read('bundle_manifest.json'))
    if (m.get('schema'),m.get('kind'),m.get('phase'))!=('h3v2_bundle_v1','inputs',PHASE): raise ValueError('Sai input bundle/phase')
    entries=m['entries']
    names=[e['path'] for e in entries]
    if len({n.casefold() for n in names})!=len(names) or sorted(z.namelist())!=sorted(names+['bundle_manifest.json']): raise ValueError('Archive members không hợp lệ')
    for e in entries:
        name=e['path'];p=Path(name)
        if p.is_absolute() or '..' in p.parts or ':' in name or chr(92) in name: raise ValueError('Unsafe path')
        if not (name.startswith(('grace_h1/','grace_h3v2/','configs/')) or name.startswith('requirements')): continue
        target=ROOT/name;target.resolve().relative_to(ROOT.resolve())
        content=z.read(name)
        if len(content)!=e['bytes'] or hashlib.sha256(content).hexdigest()!=e['sha256']: raise ValueError('Source checksum mismatch')
        if target.exists() and target.read_bytes()!=content: raise ValueError('Workspace source khác input; dùng runtime/workspace mới')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(content)
sys.path.insert(0,str(ROOT))
from grace_h3v2.portable import import_bundle,create_bundle
print(import_bundle(ROOT,INPUT,'inputs',PHASE))
print(import_bundle(ROOT,RUNTIME,'joern_runtime',PHASE))
if CHECKPOINT.exists(): print(import_bundle(ROOT,CHECKPOINT,'checkpoint',PHASE))


## Dependencies CPU và Java local

Mặc định không cài package hay tải Java trong notebook này để các bước CPU chạy offline. Full-pipeline cài dependency nếu cần; bật INSTALL_DEPENDENCIES chỉ khi bạn chủ động có mạng. Nếu package vừa đổi phiên bản, cần restart kernel local rồi chạy lại.


In [ ]:
if INSTALL_DEPENDENCIES:
    from importlib.metadata import version, PackageNotFoundError
    runtime_packages=('numpy','scipy','scikit-learn','joblib','threadpoolctl','PyYAML','rapidfuzz','python-dotenv','pandas')
    def _runtime_versions():
        result={}
        for package in runtime_packages:
            try: result[package]=version(package)
            except PackageNotFoundError: result[package]=None
        return result
    versions_before=_runtime_versions()
    subprocess.check_call([sys.executable,'-m','pip','install','--break-system-packages','-r',str(ROOT/'requirements-v2.txt')])
    if versions_before!=_runtime_versions(): raise RuntimeError('Package version changed. Restart this local notebook kernel, then run all again from the first cell.')
java_home=Path(os.environ.get('JAVA_HOME','/usr/lib/jvm/java-21-openjdk-amd64'))
java_bin=java_home/'bin/java'
if not java_bin.exists():
    java_path=shutil.which('java')
    if java_path: java_bin=Path(java_path).resolve(); java_home=java_bin.parent.parent
if not java_bin.exists(): raise FileNotFoundError('Joern cần Java 21 đã cài local. Cài Java trước khi ngắt mạng; notebook không gọi apt/download.')
os.environ['JAVA_HOME']=str(java_home)
os.environ['PATH']=str(java_home/'bin')+os.pathsep+os.environ['PATH']
os.environ['JOERN_HOME']=str(ROOT/'tools/joern/joern-cli')
for name in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'): os.environ[name]='4'
from threadpoolctl import threadpool_limits
threadpool_limits(limits=4)
from grace_h3v2 import workflow as w
CFG=w.config(ROOT)
print('Datasets:',CFG['datasets'],'seeds:',CFG['seeds'])
print('Main objective:',CFG['selection'])


## Fit và checkpoint local

Base: nested train OOF, vocabulary/scaler fit riêng trong fold, fusion học từ OOF trong outer train; không chọn classifier bằng test.

Meta: classifier được khóa, GRACE validation nằm ngoài test, meta-model dùng3fold OOF để chọn recipe/cutoff rồi refit eligible validation. Validation đã dùng học meta-model nên không gọi điểm refit là independent validation.

Checkpoint lưu trực tiếp vào bundles/ trên máy. Không phụ thuộc Drive hoặc phiên Colab.


In [ ]:
last_checkpoint={'time':time.monotonic(),'events':0}
def checkpoint(force=False):
    last_checkpoint['events']+=1
    due=last_checkpoint['events']%5==0 or time.monotonic()-last_checkpoint['time']>=300
    if not (force or due): return
    path=CHECKPOINT
    print(create_bundle(ROOT,path,'checkpoint',PHASE))
    print('Checkpoint local:',path)
    last_checkpoint['time']=time.monotonic()
try:
    result=w.base_prepare(ROOT,CFG,checkpoint) if PHASE=='base' else w.meta_prepare(ROOT,CFG,checkpoint)
finally:
    checkpoint(force=True)
print(json.dumps(result,indent=2))


## Xuất kết quả local và quay về entry

CPU output và checkpoint nằm trong **bundles/** của project. Sau khi tạo output, quay về notebooks/full-pipeline.ipynb và Run All lại:

- base-output → local preflight/inference validation → tạo meta-inputs.
- meta-output → local preflight/inference test → predict/evaluate.

Không cần chuyển file qua Drive; output zip đã nằm sẵn trong project.


In [ ]:
OUTPUT=BUNDLES/f'grace-h3v2-{PHASE}-output.zip'
info=create_bundle(ROOT,OUTPUT,'output',PHASE)
print(json.dumps(info,indent=2))
print('Output local:',OUTPUT)
print('Quay về notebooks/full-pipeline.ipynb local và Run All lại.')
